# Inspect Ingested Data Tables

This notebook helps you inspect the SQLite bronze tables used by Skimmer.

- Default DB path: `data/skimmer.db`
- Override with env var: `SKIMMER_DB_PATH`
- It lists tables, shows schema, row counts, and recent records.

In [10]:
import json
import os
import sqlite3
from pathlib import Path

import pandas as pd
from IPython.display import display

DEFAULT_DB_PATH = Path('data/skimmer.db')
DB_PATH = Path(os.environ.get('SKIMMER_DB_PATH', DEFAULT_DB_PATH)).expanduser()

print(f'Using database: {DB_PATH.resolve()}')
if not DB_PATH.exists():
    raise FileNotFoundError(f'Database not found: {DB_PATH}')

Using database: /home/orangepi/code_projects/skimmer/data/skimmer.db


In [11]:
connection = sqlite3.connect(DB_PATH)

tables_df = pd.read_sql_query(
    """
    SELECT name AS table_name
    FROM sqlite_master
    WHERE type = 'table'
    ORDER BY name
    """,
    connection,
)

if tables_df.empty:
    raise RuntimeError('No tables found in the database.')

display(tables_df)

,table_name
0,bronze_socialblade_channel_profiles
1,bronze_socialblade_channel_stats
2,bronze_vidiq_channel_profiles
3,bronze_vidiq_channel_stats
4,bronze_youtube_skimmed
5,bronze_youtubeapi_channel_stats
6,bronze_youtubeapi_video_stats
7,collection_attempts
8,collection_errors
9,profile_queue


In [12]:
def table_schema(table_name: str) -> pd.DataFrame:
    return pd.read_sql_query(f"PRAGMA table_info({table_name})", connection)


def row_count(table_name: str) -> int:
    query = f"SELECT COUNT(*) AS row_count FROM {table_name}"
    return int(pd.read_sql_query(query, connection).iloc[0]['row_count'])


def preview_table(table_name: str, limit: int = 25) -> pd.DataFrame:
    schema = table_schema(table_name)
    column_names = schema['name'].tolist()
    order_column = next(
        (
            column
            for column in ('id', 'last_seen_at', 'observed_at', 'occurred_at', 'created_at', 'latest_video_at')
            if column in column_names
        ),
        next((column for column, is_primary_key in zip(schema['name'], schema['pk']) if is_primary_key), None),
    )
    order_by = f' ORDER BY "{order_column}" DESC' if order_column else ''
    query = f'SELECT * FROM "{table_name}"{order_by} LIMIT {int(limit)}'
    df = pd.read_sql_query(query, connection)

    if 'raw_record_json' in df.columns:
        def parse_raw(value):
            if value is None:
                return None
            try:
                return json.loads(value)
            except (json.JSONDecodeError, TypeError):
                return value

        df['raw_record_json_parsed'] = df['raw_record_json'].map(parse_raw)

    return df


In [13]:
summary_rows = []
for table_name in tables_df['table_name'].tolist():
    summary_rows.append({'table_name': table_name, 'row_count': row_count(table_name)})

summary_df = pd.DataFrame(summary_rows).sort_values(['row_count', 'table_name'], ascending=[False, True])
display(summary_df)

,table_name,row_count
4,bronze_youtube_skimmed,18890
7,collection_attempts,5582
6,bronze_youtubeapi_video_stats,5482
9,profile_queue,2802
3,bronze_vidiq_channel_stats,1828
8,collection_errors,780
2,bronze_vidiq_channel_profiles,471
1,bronze_socialblade_channel_stats,311
5,bronze_youtubeapi_channel_stats,211
10,youtube_api_quota_usage,1


In [14]:
for table_name in tables_df['table_name'].tolist():
    print('\n' + '=' * 100)
    print(f'TABLE: {table_name}')
    print('=' * 100)

    print('\nSchema:')
    display(table_schema(table_name))

    print('\nLatest rows:')
    display(preview_table(table_name, limit=20))


TABLE: bronze_socialblade_channel_profiles

Schema:


,cid,name,type,notnull,dflt_value,pk
0,0,id,INTEGER,0,None,1
1,1,channel_id,TEXT,1,None,0
2,2,channel_name,TEXT,0,None,0
3,3,channel_type,TEXT,0,None,0
4,4,country,TEXT,0,None,0
5,5,created_at,TEXT,0,None,0
6,6,subscribers_total,,0,None,0
7,7,views_total,,0,None,0
8,8,videos_total,,0,None,0
9,9,data_digest,TEXT,1,None,0



Latest rows:


,id,channel_id,channel_name,channel_type,country,created_at,subscribers_total,views_total,videos_total,data_digest



TABLE: bronze_socialblade_channel_stats

Schema:


...

,channel_key,channel_id,channel_name,latest_video_at,last_seen_at,last_success_at,digested,assigned_source,vidiq_failed,socialblade_failed,needs_review,claimed_by,claimed_at,youtube_channel_id,channel_id_claimed_by,channel_id_claimed_at,youtube_channel_id_attempted_at,youtube_api_failed,youtube_api_attempted_at,youtube_api_success_at
0,@abcnews,@ABCNews,ABC News,2026-07-23T02:10:39+00:00,2026-07-23T03:10:39+00:00,2026-07-23T03:01:11+00:00,1,vidiq,1,1,0,None,None,UCBi2mrWuNuyYy4gbM6fU18Q,None,None,None,0,2026-07-23T02:54:09+00:00,2026-07-23T03:01:11+00:00
1,@airrack,@airrack,Airrack,2026-07-17T07:34:11+00:00,2026-07-23T03:10:39+00:00,2026-07-20T12:12:27+00:00,0,vidiq,0,1,0,vidiq-18270,2026-07-23T02:23:21+00:00,None,None,None,None,1,2026-07-23T02:21:25+00:00,None
2,@akstareng,@AKSTARENG,AKSTAR ENG,2026-04-23T23:57:27+00:00,2026-07-23T03:10:39+00:00,None,1,socialblade,1,1,1,youtube-api-24543,2026-07-23T02:54:09+00:00,UCIAie1orP_SjI0T591dAUjw,None,None,None,1,2026-07-23T02:54:09+00:00,None
3,@allanwv,@allanwv,allanwv,2009-07-27T03:10:39+00:00,2026-07-23T03:10:39+00:00,2026-07-20T03:21:47+00:00,1,vidiq,0,0,0,youtube-api-24543,2026-07-23T02:54:09+00:00,UCcQQ3J0uzK4GO6BKF7KIYKA,None,None,None,1,2026-07-23T02:54:09+00:00,None
4,@andreijikh,@AndreiJikh,Andrei Jikh,2026-07-21T15:18:55+00:00,2026-07-23T03:10:39+00:00,2026-07-23T03:02:18+00:00,1,vidiq,1,1,0,None,None,UCGy7SkBjcIAgTiwkXEtPnYg,None,None,None,0,2026-07-23T02:54:09+00:00,2026-07-23T03:02:18+00:00
5,@andreipiano,@AndreiPiano,Andrei Piano,2022-07-24T03:10:39+00:00,2026-07-23T03:10:39+00:00,2026-07-20T03:07:04+00:00,1,vidiq,0,0,0,youtube-api-24543,2026-07-23T02:54:09+00:00,UC6oPj0CXwGi1M280hUW-vyw,None,None,None,1,2026-07-23T02:54:09+00:00,None
6,@annaaaltamirano,@annaaaltamirano,Anna Altamirano,2026-07-20T16:08:02+00:00,2026-07-23T03:10:39+00:00,2026-07-22T05:19:34+00:00,1,vidiq,1,1,1,youtube-api-24543,2026-07-23T02:54:09+00:00,UCA-0l1VBIG1uTvVeMD_l5kw,None,None,None,1,2026-07-23T02:54:09+00:00,None
7,@asmongoldclips,@AsmongoldClips,Asmongold Clips,2026-07-19T17:33:42+00:00,2026-07-23T03:10:39+00:00,2026-07-22T00:37:52+00:00,0,socialblade,1,0,0,youtube-api-24543,2026-07-23T02:54:09+00:00,UCMwJJL5FJFuTRT55ksbQ4GQ,None,None,2026-07-22T06:08:34+00:00,1,2026-07-23T02:54:09+00:00,None
8,@asmontv,@AsmonTV,Asmongold TV,2026-07-22T23:02:27+00:00,2026-07-23T03:10:39+00:00,2026-07-21T03:01:29+00:00,0,socialblade,1,0,0,socialblade-1718,2026-07-23T01:21:27+00:00,UCQeRaTukNYft1_6AZPACnog,None,None,2026-07-21T03:57:55+00:00,0,None,None
9,@beardmeatsfood,@Beardmeatsfood,BeardMeatsFood,2026-07-19T17:33:42+00:00,2026-07-23T03:10:39+00:00,2026-07-21T06:18:26+00:00,1,vidiq,1,1,1,youtube-api-24543,2026-07-23T02:54:09+00:00,UCc9CjaAjsMMvaSghZB7-Kog,None,None,None,1,2026-07-23T02:54:09+00:00,None



TABLE: youtube_api_quota_usage

Schema:


,cid,name,type,notnull,dflt_value,pk
0,0,quota_date,TEXT,0,None,1
1,1,units_used,INTEGER,1,0,0



Latest rows:


,quota_date,units_used
0,2026-07-22,2650


In [9]:
connection.close()

In [16]:
display(preview_table('profile_queue'))

,channel_key,channel_id,channel_name,latest_video_at,last_seen_at,last_success_at,digested,assigned_source,vidiq_failed,socialblade_failed,needs_review,claimed_by,claimed_at,youtube_channel_id,channel_id_claimed_by,channel_id_claimed_at,youtube_channel_id_attempted_at,youtube_api_failed,youtube_api_attempted_at,youtube_api_success_at
0,@abcnews,@ABCNews,ABC News,2026-07-23T02:10:39+00:00,2026-07-23T03:10:39+00:00,2026-07-23T03:01:11+00:00,1,vidiq,1,1,0,None,None,UCBi2mrWuNuyYy4gbM6fU18Q,None,None,None,0,2026-07-23T02:54:09+00:00,2026-07-23T03:01:11+00:00
1,@airrack,@airrack,Airrack,2026-07-17T07:34:11+00:00,2026-07-23T03:10:39+00:00,2026-07-20T12:12:27+00:00,0,vidiq,0,1,0,vidiq-18270,2026-07-23T02:23:21+00:00,None,None,None,None,1,2026-07-23T02:21:25+00:00,None
2,@akstareng,@AKSTARENG,AKSTAR ENG,2026-04-23T23:57:27+00:00,2026-07-23T03:10:39+00:00,None,1,socialblade,1,1,1,youtube-api-24543,2026-07-23T02:54:09+00:00,UCIAie1orP_SjI0T591dAUjw,None,None,None,1,2026-07-23T02:54:09+00:00,None
3,@allanwv,@allanwv,allanwv,2009-07-27T03:10:39+00:00,2026-07-23T03:10:39+00:00,2026-07-20T03:21:47+00:00,1,vidiq,0,0,0,youtube-api-24543,2026-07-23T02:54:09+00:00,UCcQQ3J0uzK4GO6BKF7KIYKA,None,None,None,1,2026-07-23T02:54:09+00:00,None
4,@andreijikh,@AndreiJikh,Andrei Jikh,2026-07-21T15:18:55+00:00,2026-07-23T03:10:39+00:00,2026-07-23T03:02:18+00:00,1,vidiq,1,1,0,None,None,UCGy7SkBjcIAgTiwkXEtPnYg,None,None,None,0,2026-07-23T02:54:09+00:00,2026-07-23T03:02:18+00:00
5,@andreipiano,@AndreiPiano,Andrei Piano,2022-07-24T03:10:39+00:00,2026-07-23T03:10:39+00:00,2026-07-20T03:07:04+00:00,1,vidiq,0,0,0,youtube-api-24543,2026-07-23T02:54:09+00:00,UC6oPj0CXwGi1M280hUW-vyw,None,None,None,1,2026-07-23T02:54:09+00:00,None
6,@annaaaltamirano,@annaaaltamirano,Anna Altamirano,2026-07-20T16:08:02+00:00,2026-07-23T03:10:39+00:00,2026-07-22T05:19:34+00:00,1,vidiq,1,1,1,youtube-api-24543,2026-07-23T02:54:09+00:00,UCA-0l1VBIG1uTvVeMD_l5kw,None,None,None,1,2026-07-23T02:54:09+00:00,None
7,@asmongoldclips,@AsmongoldClips,Asmongold Clips,2026-07-19T17:33:42+00:00,2026-07-23T03:10:39+00:00,2026-07-22T00:37:52+00:00,0,socialblade,1,0,0,youtube-api-24543,2026-07-23T02:54:09+00:00,UCMwJJL5FJFuTRT55ksbQ4GQ,None,None,2026-07-22T06:08:34+00:00,1,2026-07-23T02:54:09+00:00,None
8,@asmontv,@AsmonTV,Asmongold TV,2026-07-22T23:02:27+00:00,2026-07-23T03:10:39+00:00,2026-07-21T03:01:29+00:00,0,socialblade,1,0,0,socialblade-1718,2026-07-23T01:21:27+00:00,UCQeRaTukNYft1_6AZPACnog,None,None,2026-07-21T03:57:55+00:00,0,None,None
9,@beardmeatsfood,@Beardmeatsfood,BeardMeatsFood,2026-07-19T17:33:42+00:00,2026-07-23T03:10:39+00:00,2026-07-21T06:18:26+00:00,1,vidiq,1,1,1,youtube-api-24543,2026-07-23T02:54:09+00:00,UCc9CjaAjsMMvaSghZB7-Kog,None,None,None,1,2026-07-23T02:54:09+00:00,None
